In [5]:
from collections import Counter
from datasets import concatenate_datasets, load_dataset

# Download dataset
_ds = load_dataset("talzoomanzoo/openr1-aime-train-test")

# Show counts per split
for split_name, split_ds in _ds.items():
    sources = split_ds["data_source"]
    counts = Counter(sources)
    print(f"\n{split_name} (rows={len(split_ds)}):")
    for key, value in sorted(counts.items()):
        print(f"  {key}: {value}")

# Overall counts
all_counts = Counter()
for split_ds in _ds.values():
    all_counts.update(split_ds["data_source"])
print("\nall_splits:")
for key, value in sorted(all_counts.items()):
    print(f"  {key}: {value}")

# Filter (train only): keep all AIME-related sources, plus 5000 random from others
keep_sources = {"aime", "aime25", "amc_aime"}
seed = 42

train_ds = _ds["train"]
keep_ds = train_ds.filter(lambda x: x["data_source"] in keep_sources)
other_ds = train_ds.filter(lambda x: x["data_source"] not in keep_sources)

sample_size = min(5000, len(other_ds))
other_sample = other_ds.shuffle(seed=seed).select(range(sample_size))
filtered_ds = concatenate_datasets([keep_ds, other_sample]).shuffle(seed=seed)

print(f"\nkeep_sources rows (train): {len(keep_ds)}")
print(f"other_sample rows (train): {len(other_sample)}")
print(f"filtered_ds rows (train): {len(filtered_ds)}")
print("filtered_ds data_source counts:")
filtered_counts = Counter(filtered_ds["data_source"])
for key, value in sorted(filtered_counts.items()):
    print(f"  {key}: {value}")

# Upload to Hugging Face Hub (train + original test)
# Ensure you are logged in: `huggingface-cli login` or set HF_TOKEN env var.
from datasets import DatasetDict

repo_id = "talzoomanzoo/openr1-aime-train-test-5k"
dataset_to_push = DatasetDict({
    "train": filtered_ds,
    "test": _ds["test"],
})
dataset_to_push.push_to_hub(repo_id)



train (rows=45822):
  aime: 15
  aime25: 15
  amc_aime: 767
  aops_forum: 3850
  cn_contest: 7836
  inequalities: 386
  number_theory: 143
  olympiads: 32626
  olympiads_ref: 184

test (rows=1920):
  aime: 960
  aime25: 960

all_splits:
  aime: 975
  aime25: 975
  amc_aime: 767
  aops_forum: 3850
  cn_contest: 7836
  inequalities: 386
  number_theory: 143
  olympiads: 32626
  olympiads_ref: 184

keep_sources rows (train): 797
other_sample rows (train): 5000
filtered_ds rows (train): 5797
filtered_ds data_source counts:
  aime: 15
  aime25: 15
  amc_aime: 767
  aops_forum: 426
  cn_contest: 866
  inequalities: 45
  number_theory: 20
  olympiads: 3624
  olympiads_ref: 19


Creating parquet from Arrow format: 100%|██████████| 6/6 [00:00<00:00, 26.04ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Upload 0 LFS files: 0it [00:00, ?it/s]
Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 911.01ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.40s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/openr1-aime-train-test-5k/commit/9b7271dc9a56e96a3140a7a811471fd3534c3fa1', commit_message='Upload dataset', commit_description='', oid='9b7271dc9a56e96a3140a7a811471fd3534c3fa1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/openr1-aime-train-test-5k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/openr1-aime-train-test-5k'), pr_revision=None, pr_num=None)